## FFC — Step 2 (Local)

This notebook is a 1:1 local execution version of `FFC/FFC Test/Step_2_2026_Offtake_Primary_Data_Processing_LOCAL.py`.

### Inputs (local)
- `output/offtake_combined_local_march_2026.csv` (from Step 1 local)
- `input/2026_customer_abbott_code_mapping.xlsx`
- `input/2026_Consolidated_Price_Master.xlsx`
- `input/2025_Conversion_Master.xlsx`
- `input/2026_SKU_Brand_Division_mapping.xlsx`
- `input/2026_PinCodes_with_States_and_Districts.csv`
- `input/2026_EAP_Primary_Sales.csv`

### Outputs
- `output/offtake_step2_final.csv`
- `output/qc_step2.xlsx`


In [1]:
import os
from pathlib import Path
import logging
import pandas as pd
import numpy as np

# LOGGING

os.makedirs("logs", exist_ok=True)
os.makedirs("output", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("logs/step2_run.log"),
        logging.StreamHandler(),
    ],
)

logger = logging.getLogger(__name__)

# INPUT FILES

STEP1_FILE = Path("output/offtake_combined_local_march_2026.csv")

PRODUCT_MAP_FILE = Path("input/2026_customer_abbott_code_mapping.xlsx")
PRICE_FILE = Path("input/2026_Consolidated_Price_Master.xlsx")
CONVERSION_FILE = Path("input/2025_Conversion_Master.xlsx")
BRAND_DIV_FILE = Path("input/2026_SKU_Brand_Division_mapping.xlsx")
PINCODE_FILE = Path("input/2026_PinCodes_with_States_and_Districts.csv")
PRIMARY_FILE = Path("input/2026_EAP_Primary_Sales.csv")

for p in [STEP1_FILE, PRODUCT_MAP_FILE, PRICE_FILE, CONVERSION_FILE, BRAND_DIV_FILE, PINCODE_FILE, PRIMARY_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")


In [2]:

# LOAD STEP‑1 OFFTAKE

logger.info("Loading Step-1 Offtake")

offtake = pd.read_csv(
    STEP1_FILE,
    parse_dates=["month"],
    dtype={"product_code": "string", "pincode": "string"},
    low_memory=False,
)

offtake["name of customer"] = offtake["name of customer"].str.upper()

# CRITICAL: create ECB_Primary_Sales column EARLY
offtake["ECB_Primary_Sales"] = 0.0

offtake.shape


2026-05-12 17:28:20,316 | INFO | Loading Step-1 Offtake


(124074, 9)

In [3]:
# offtake['channel_sales'].sum()

In [4]:

# LOAD MASTER FILES

logger.info("Loading mapping masters")

product_map = pd.read_excel(PRODUCT_MAP_FILE)
price_master = pd.read_excel(PRICE_FILE)
conversion_master = pd.read_excel(CONVERSION_FILE)
brand_div = pd.read_excel(BRAND_DIV_FILE)

pincode_map = pd.read_csv(
    PINCODE_FILE,
    encoding="latin1",
    dtype={"Pin.Code": "string"},
    low_memory=False,
)

logger.info("Masters loaded")
{
    "product_map": product_map.shape,
    "price_master": price_master.shape,
    "conversion_master": conversion_master.shape,
    "brand_div": brand_div.shape,
    "pincode_map": pincode_map.shape,
}


2026-05-12 17:28:20,528 | INFO | Loading mapping masters
2026-05-12 17:28:25,125 | INFO | Masters loaded


{'product_map': (19813, 3),
 'price_master': (72766, 3),
 'conversion_master': (9937, 3),
 'brand_div': (2492, 6),
 'pincode_map': (20873, 3)}

In [5]:

# LOAD & NORMALIZE PRIMARY SALES

logger.info("Loading ECB Primary Sales")

primary = pd.read_csv(PRIMARY_FILE, low_memory=False)

primary["month"] = pd.to_datetime(primary["month"], dayfirst=True, errors="coerce")

# Detect primary sales column
primary_col = None
for col in primary.columns:
    norm = col.strip().replace(" ", "_").upper()
    if norm in ["ECB_PRIMARY_SALES", "ECBPRIMARYSALES", "PRIMARY_SALES"]:
        primary_col = col
        break

if primary_col is None:
    raise ValueError("ECB Primary Sales column not found")

primary["ECB_Primary_Sales"] = pd.to_numeric(primary[primary_col], errors="coerce").fillna(0)

primary.shape


2026-05-12 17:28:25,140 | INFO | Loading ECB Primary Sales
C:\Users\pawarux2\AppData\Local\Temp\ipykernel_2900\3584306490.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  primary["month"] = pd.to_datetime(primary["month"], dayfirst=True, errors="coerce")


(79673, 12)

In [6]:

# NORMALIZATION

product_map["Customer"] = product_map["Customer"].str.upper()
conversion_master["Customer"] = conversion_master["Customer"].str.upper()

product_map["SKU No"] = pd.to_numeric(product_map["SKU No"], errors="coerce").fillna(product_map["SKU No"])
price_master["sap_code"] = pd.to_numeric(price_master["sap_code"], errors="coerce").fillna(price_master["sap_code"])
brand_div["SKU Code"] = pd.to_numeric(brand_div["SKU Code"], errors="coerce").fillna(brand_div["SKU Code"])

logger.info("Normalization completed")


2026-05-12 17:28:25,366 | INFO | Normalization completed


In [7]:
# 1️ CUSTOMER SKU → ABBOTT SKU

logger.info("Mapping Customer SKU to Abbott SKU")

offtake = offtake.merge(
    product_map,
    how="left",
    left_on=["product_code", "name of customer"],
    right_on=["Customer Item Code", "Customer"],
)

offtake.shape


2026-05-12 17:28:25,380 | INFO | Mapping Customer SKU to Abbott SKU


(124074, 12)

In [8]:
def clean_key_col(s):
    return (
        s.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

# Customer names
offtake["name of customer"] = clean_key_col(offtake["name of customer"])
product_map["Customer"] = clean_key_col(product_map["Customer"])
conversion_master["Customer"] = clean_key_col(conversion_master["Customer"])

# Product code keys
offtake["product_code"] = clean_key_col(offtake["product_code"])
product_map["Customer Item Code"] = clean_key_col(product_map["Customer Item Code"])
conversion_master["Channel Product Code"] = clean_key_col(conversion_master["Channel Product Code"])

# SKU keys
product_map["SKU No"] = clean_key_col(product_map["SKU No"])
brand_div["SKU Code"] = clean_key_col(brand_div["SKU Code"])
price_master["sap_code"] = clean_key_col(price_master["sap_code"])

# Month keys
offtake["month"] = pd.to_datetime(offtake["month"], errors="coerce").dt.to_period("M").dt.to_timestamp()
price_master["month"] = pd.to_datetime(price_master["month"], errors="coerce").dt.to_period("M").dt.to_timestamp()

In [ ]:

# 2️ PTS PRICE
logger.info("Applying PTS price")

offtake = offtake.merge(
    price_master,
    how="left",
    left_on=["SKU No", "month"],
    right_on=["sap_code", "month"],
)

offtake.shape


2026-05-12 17:28:25,912 | INFO | Applying PTS price


(124074, 14)

In [10]:
print("After SKU mapping")
print("Rows:", offtake.shape[0])
print("SKU No mapped rows:", offtake["SKU No"].notna().sum())
print("SKU No blank rows:", offtake["SKU No"].isna().sum())

print(
    offtake.assign(mapped=offtake["SKU No"].notna())
    .groupby("name of customer")["mapped"]
    .agg(["sum", "count"])
)

After SKU mapping
Rows: 124074
SKU No mapped rows: 55615
SKU No blank rows: 68459
                    sum  count
name of customer              
NETMEDS               0  28399
TRUEMEDS          55615  58098
WELLNESS              0  37577


In [11]:
# Clean up any previous conversion merge (safe to run multiple times)
cols_to_drop = [
    "Customer",
    "Channel Product Code",
    "Conversion Factor",
]

offtake = offtake.drop(
    columns=[c for c in cols_to_drop if c in offtake.columns],
    errors="ignore",
)

In [12]:
logger.info("Applying conversion factor")

offtake = offtake.merge(
    conversion_master,
    how="left",
    left_on=["name of customer", "product_code"],
    right_on=["Customer", "Channel Product Code"],
)

offtake["units_sold"] = pd.to_numeric(offtake["units_sold"], errors="coerce")
cond = offtake["Conversion Factor"].notna()

offtake["final_converted_quantity"] = np.where(
    cond,
    offtake["units_sold"] / offtake["Conversion Factor"],
    offtake["units_sold"],
)

# ✅ LOCAL PIPELINE FIX:
# Step-1 revenue is already channel sales
offtake["channel_sales"] = pd.to_numeric(offtake["revenue"], errors="coerce").fillna(0)

# Keep revenue as GMV for final output
offtake.rename(columns={"revenue": "GMV"}, inplace=True)

offtake[["units_sold", "channel_sales", "GMV"]].head()

2026-05-12 17:28:26,033 | INFO | Applying conversion factor


,units_sold,channel_sales,GMV
0,5.0,23.50,23.50
1,4.0,23.50,23.50
2,1.0,28.67,28.67
3,1.0,259.11,259.11
4,2.0,62.29,62.29


In [13]:
# 4️ APPEND PRIMARY SALES

logger.info("Appending ECB Primary Sales")

primary_cols = offtake.columns
primary = primary.reindex(columns=primary_cols, fill_value=0)

offtake = pd.concat([offtake, primary], axis=0, ignore_index=True)

offtake.shape


2026-05-12 17:28:26,109 | INFO | Appending ECB Primary Sales


(203747, 18)

In [21]:
# 5️ BRAND / DIVISION

logger.info("Applying Brand & Division")

offtake = offtake.merge(
    brand_div,
    how="left",
    left_on="SKU No",
    right_on="SKU Code",
)

offtake.shape


2026-05-12 17:30:46,753 | INFO | Applying Brand & Division


(203747, 33)

In [15]:
# 6️ PINCODE MAP

logger.info("Applying Pincode to State/District")

offtake = offtake.merge(
    pincode_map,
    how="left",
    left_on="pincode",
    right_on="Pin.Code",
)

offtake.shape


2026-05-12 17:28:26,226 | INFO | Applying Pincode to State/District


(203747, 27)

In [16]:
# 7️ BUSINESS OVERRIDES

override_customers = [
    "ASTER",
    "ASWAS",
    "NOBLE",
    "DAWADOST",
    "SASTA AROGYA",
    "GUARDIAN",
    "THULASI",
    "ZENO HEALTH",
    "EASYMEDICO",
]

mask = offtake["name of customer"].isin(override_customers)
offtake.loc[mask, "channel_sales"] = offtake.loc[mask, "ECB_Primary_Sales"]

logger.info("Overrides applied")


2026-05-12 17:28:26,298 | INFO | Overrides applied


In [17]:
## 8️⃣ FINAL OUTPUT

final_cols = [
    "month",
    "name of customer",
    "pincode",
    "State",
    "District",
    "SKU No",
    "Brand Name",
    "Division Name",
    "units_sold",
    "channel_sales",
    "ECB_Primary_Sales",
    "GMV",
]

offtake_final = offtake[final_cols].copy()

# ✅ ADD AFFILIATE (NO BUSINESS LOGIC CHANGE)
offtake_final["Affiliate"] = "AIL"

# Reorder to keep hierarchy clean (optional but recommended)
offtake_final = offtake_final[
    [
        "month",
        "name of customer",
        "Affiliate",
        "pincode",
        "State",
        "District",
        "SKU No",
        "Brand Name",
        "Division Name",
        "units_sold",
        "channel_sales",
        "ECB_Primary_Sales",
        "GMV",
    ]
]

# Fill numeric nulls (unchanged logic)
for c in ["channel_sales", "ECB_Primary_Sales", "GMV"]:
    offtake_final[c] = offtake_final[c].fillna(0)

offtake_final.shape

(203747, 13)

In [18]:
# QC

qc = (
    offtake_final.groupby(["name of customer", "month"])
    .agg(
        Rows=("SKU No", "count"),
        Channel_Sales=("channel_sales", "sum"),
        ECB_Primary=("ECB_Primary_Sales", "sum"),
        GMV=("GMV", "sum"),
    )
    .reset_index()
)

qc


,name of customer,month,Rows,Channel_Sales,ECB_Primary,GMV
0,Amazon & Flipkart,2026-01-01,751,0.00,1.479812e+07,0.00
1,Amazon & Flipkart,2026-02-01,775,0.00,1.165204e+07,0.00
2,Apollo Healthco,2026-01-01,346,0.00,3.179701e+06,0.00
3,Apollo Healthco,2026-02-01,492,0.00,4.619675e+06,0.00
4,Apollo Retail,2026-01-01,8495,0.00,2.275971e+08,0.00
5,Apollo Retail,2026-02-01,9878,0.00,3.038316e+08,0.00
6,Aster,2026-01-01,1908,0.00,1.309119e+07,0.00
7,Aster,2026-02-01,1088,0.00,1.239904e+07,0.00
8,Aswas,2026-01-01,502,0.00,1.320813e+07,0.00
9,Aswas,2026-02-01,345,0.00,1.174087e+07,0.00


In [19]:
offtake_final["name of customer"] = (
    offtake_final["name of customer"]
    .str.upper()
    .str.strip()
    .replace({
        "APOLLO RETAIL": "APOLLO",
        "KEIMED": "KEIMEDGT",
    })
)

In [20]:
# WRITE OUTPUT

offtake_final.to_csv("output/offtake_step2_final.csv", index=False)

with pd.ExcelWriter("output/qc_step2.xlsx") as writer:
    qc.to_excel(writer, sheet_name="Account_Month_QC", index=False)

logger.info("Step-2 local notebook completed successfully")
{
    "final_csv": "output/offtake_step2_final.csv",
    "qc_xlsx": "output/qc_step2.xlsx",
}


2026-05-12 17:28:27,782 | INFO | Step-2 local notebook completed successfully


{'final_csv': 'output/offtake_step2_final.csv',
 'qc_xlsx': 'output/qc_step2.xlsx'}